In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from unstructured.staging.base import elements_to_json, elements_from_json
from unstructured.chunking.title import chunk_by_title
from src.document_parsing import parse_document, \
                                    save_parsed_elements_local, \
                                    save_parsed_elements, \
                                    save_chunks,\
                                    chunks_to_dataframe
from src.db import get_connection
from src.embeddings import chunks_to_vectors, save_vector_embeddings
from fastprogress.fastprogress import master_bar, progress_bar
import os
import json
import pandas as pd

In [3]:
from pathlib import Path

RAW_DIR = Path("../data/raw/finance/")
PARSED_DIR = "../data/parsed/"

full_paths = list(RAW_DIR.iterdir())[:10]
print(f"File Length: ({len(full_paths)})")

File Length: (10)


In [4]:
raw_parsing_data = []
for file_path in progress_bar(full_paths):
    parsing_elements = parse_document(file_path, infer_table_structure=True)
    raw_parsing_data.append(parsing_elements)

#save_parsed_elements_local(PARSED_DIR, raw_parsing_data)

<div><progress max="10" value="10"></progress> 100.00% [10/10 04:46&lt;00:00]</div>

No languages specified, defaulting to English.


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.


In [8]:
conn = get_connection()
documents_df = save_parsed_elements(conn, raw_parsing_data)
conn.close()

In [19]:
document_chunks_data = []
for document_elements in raw_parsing_data:
    document_chunks = chunk_by_title(
        document_elements,
        max_characters=2000,              # hard ceiling per chunk
        combine_text_under_n_chars=200,   # merge tiny chunks into neighbors
        new_after_n_chars=200,            # soft target before starting a new chunk
        multipage_sections=True,          # allow a section to span page breaks
    )
    document_chunks_data.append(document_chunks)

In [15]:
conn = get_connection()
chunks_df = save_chunks(conn, document_chunks_data)
conn.close()

In [21]:
vectors_df = chunks_to_vectors(chunks_df)

In [ ]:
conn = get_connection()
vector_embeddings_df = save_vector_embeddings(conn, vectors_df)
conn.close()